# Mainstream LLM Performance Comparison

Directly use the zero-shot prompting method to test the performance of mainstream LLMs. The prompt used for the request is consistent with the official testing method:
```
prompt + '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'
```

The test results are as follows:

| model | think | max_tokens | score |
| --- | --- | --- | --- |
| Gemini-3.1-Pro | ✅ | 32768 | **0.81** |
| Claude-Opus-4.6 | ✅ | 32768 | **0.78** |
| DeepSeek-V3.2   | ✅ | 32768 | **0.74** |
| Kimi-K2.5 | ✅ | 32768 | 0.72 |
| Qwen3-Max | ✅ | 32768 | 0.72 |
| MiniMax-M2.5 | ✅ | 32768 | 0.66 |
| Qwen3.5-Plus | ✅ | 32768 | 0.64 |
| GLM-5 | ✅ | 32768 | 0.52 | 0.52 |
| Claude-Sonnet-4.5 | ❌ | 32768 | 0.51 |
| GPT-5.4 | ❌ | 32768 | 0.36 |

## Load Data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from IPython.display import display, HTML

In [ ]:
def highlight_text(text):
    display(HTML(f'<span style="background-color: red; color: white; padding: 2px 5px; font-weight: bold;">{text}</span>'))

In [ ]:
sample_df = pd.read_json('/kaggle/input/datasets/jiazhuang/nvidia-nemotron-train-sample-llm-result/train_sample_llm_result.jsonl', lines=True)

In [ ]:
sample_df.head()

In [ ]:
r = sample_df.sample().iloc[0]

In [ ]:
highlight_text('prompt:')
print(r.prompt)
print()

highlight_text(f'answer: {r.answer}')
print()

for model, output in r['llm_output'].items():
    highlight_text(model)
    print(output['response'])
    print()

## Extract Answer and Calc Score

In [ ]:
# Refer: https://www.kaggle.com/code/metric/nvidia-nemotron-metric

import re

def extract_final_answer(text: str | None) -> str:
    r"""Extracts the final answer from the model response.

    Prioritizes extracting answers inside `\boxed{}`.
    If no `\boxed{}` format is found, attempts to extract numbers from other formats.

    Examples:
        >>> extract_final_answer(r"The answer is \boxed{42}")
        '42'
        >>> extract_final_answer("The final answer is: 3.14")
        '3.14'
        >>> extract_final_answer("Just a number 100 in text")
        '100'
        >>> extract_final_answer(None)
        'NOT_FOUND'
    """
    if text is None:
        return 'NOT_FOUND'

    # Search for boxed answer
    # Match all instances of \boxed{...} or unclosed \boxed{ at the end
    matches = re.findall(r'\\boxed\{([^}]*)(?:\}|$)', text)
    if matches:
        non_empty = [m.strip() for m in matches if m.strip()]
        if non_empty:
            return non_empty[-1]
        return matches[-1].strip()

    # Other common formats if \boxed{} is not found
    patterns = [
        r'The final answer is:\s*([^\n]+)',
        r'Final answer is:\s*([^\n]+)',
        r'Final answer\s*[:：]\s*([^\n]+)',
        r'final answer\s*[:：]\s*([^\n]+)',
    ]
    for pattern in patterns:
        matches = re.findall(pattern, text, re.IGNORECASE)
        if matches:
            return matches[-1].strip()

    # If no structured format is found, extract the last valid number in the text
    matches = re.findall(r'-?\d+(?:\.\d+)?', text)
    if matches:
        return matches[-1]

    # If no numeric answer is found, return the last line of text as a fallback
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return lines[-1] if lines else 'NOT_FOUND'


def verify(stored_answer: str, predicted: str) -> bool:
    """Verify if the answer matches.

    For numerical answers, allow them to be judged as equal within a certain relative tolerance (1e-2);
    otherwise, compare strictly as strings (case-insensitive).
    """
    # Clean up strings
    stored_answer = stored_answer.strip()
    predicted = predicted.strip()

    try:
        # Try to convert the answers to floating point numbers
        stored_num = float(stored_answer)
        predicted_num = float(predicted)
        # Use a small absolute tolerance for numbers near zero
        return math.isclose(stored_num, predicted_num, rel_tol=1e-2, abs_tol=1e-5)
    except Exception:
        # Fallback to case-insensitive string comparison
        return predicted.lower() == stored_answer.lower()

In [ ]:
def get_extracted_answer(llm_output):
    res = {}
    
    if not isinstance(llm_output, dict): return res
    
    for model, output in llm_output.items():
        response = output.get('response')
        if response is None: continue  # skip failed request
        extracted_answer = extract_final_answer(response)
        res[model] = extracted_answer
    return res

In [ ]:
sample_df['extracted_answer'] = sample_df.llm_output.apply(get_extracted_answer)

In [ ]:
def get_accuracy_score(extracted_answer, answer):
    res = {}
    for model, extracted in extracted_answer.items():
        acc = verify(answer, extracted)
        res[model] = acc
    return res

In [ ]:
sample_df['accuracy_score'] = sample_df.apply(
    lambda r: get_accuracy_score(r.extracted_answer, r.answer),
    axis=1,
)

In [ ]:
score_df = pd.DataFrame(sample_df.accuracy_score.tolist())

In [ ]:
score_df.notna().sum()

A request failure indicates that the maximum token limit has been exceeded and **is considered an incorrect response**.

In [ ]:
score_df.fillna(False).mean().sort_values(ascending=False)

The performance of the **Claude-Opus-4.6** model falls far short of expectations; let's investigate what's going on.

In [ ]:
flag = sample_df.accuracy_score.apply(lambda dct: dct['Claude-Opus-4.6']).eq(False) & \
       sample_df.accuracy_score.apply(lambda dct: dct['DeepSeek-V3.2']).eq(True)

In [ ]:
for r in sample_df[flag].head(5).itertuples():
    print('- ' * 40)
    highlight_text('prompt:')
    print(r.prompt)
    print()

    highlight_text(f'answer: {r.answer}')
    print()

    highlight_text('Claude-Opus-4.6 Response:')
    print(r.llm_output['Claude-Opus-4.6']['response'])

    highlight_text('Claude-Opus-4.6 Extracted answer: ')
    print(r.extracted_answer['Claude-Opus-4.6'])
    print('\n\n')

It appears to be a LaTeX formatting issue. Opus generates more rigorous LaTeX code with additional formatting characters, which causes mismatches in answer validation, for example:

- alice follows above garden: \boxed{alice\ follows\ above\ garden}
- student draws the curious key: \boxed{\text{student draws the curious key}}
- 49.37: \boxed{49.37 \text{ m}}

## Update Extract Function

In [ ]:
# add process logic of: $$\boxed{\text{student draws the curious key}}$$

def extract_final_answer(text: str | None) -> str:
    r"""Extracts the final answer from the model response.

    Prioritizes extracting answers inside `\boxed{}`.
    If no `\boxed{}` format is found, attempts to extract numbers from other formats.

    Examples:
        >>> extract_final_answer(r"The answer is \boxed{42}")
        '42'
        >>> extract_final_answer("The final answer is: 3.14")
        '3.14'
        >>> extract_final_answer("Just a number 100 in text")
        '100'
        >>> extract_final_answer(None)
        'NOT_FOUND'
    """
    if text is None:
        return 'NOT_FOUND'

    ########### Add this for \boxed{\text{...}} pattern ###########
    # Search for boxed answer
    # Match all instances of \boxed{\text{...}} or unclosed \boxed{ at the end
    matches = re.findall(r'\\boxed\{\\text{([^}]*)(?:\}|$)', text)
    if matches:
        non_empty = [m.strip() for m in matches if m.strip()]
        if non_empty:
            return non_empty[-1]
        return matches[-1].str
    ################################################################    
    
    # Search for boxed answer
    # Match all instances of \boxed{...} or unclosed \boxed{ at the end
    matches = re.findall(r'\\boxed\{([^}]*)(?:\}|$)', text)
    if matches:
        non_empty = [m.strip() for m in matches if m.strip()]
        if non_empty:
            return non_empty[-1]
        return matches[-1].strip()

    # Other common formats if \boxed{} is not found
    patterns = [
        r'The final answer is:\s*([^\n]+)',
        r'Final answer is:\s*([^\n]+)',
        r'Final answer\s*[:：]\s*([^\n]+)',
        r'final answer\s*[:：]\s*([^\n]+)',
    ]
    for pattern in patterns:
        matches = re.findall(pattern, text, re.IGNORECASE)
        if matches:
            return matches[-1].strip()

    # If no structured format is found, extract the last valid number in the text
    matches = re.findall(r'-?\d+(?:\.\d+)?', text)
    if matches:
        return matches[-1]

    # If no numeric answer is found, return the last line of text as a fallback
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return lines[-1] if lines else 'NOT_FOUND'

In [ ]:
def latex_postprocess(text):
    # 'alice\\ follows\\ above\\ garden' -> 'alice follows above garden'
    text = text.replace('\\ ', ' ')
    text = text.split()
    text = ' '.join(text)
    
    # remove the units: '24.16 \\text{ m}'
    if '\\text' in text:
        text = text.split('\\text')[0].strip()
    
    return text


def get_extracted_answer(llm_output):
    res = {}
    
    if not isinstance(llm_output, dict): return res
    
    for model, output in llm_output.items():
        response = output.get('response')
        if response is None: continue  # skip failed request
        extracted_answer = extract_final_answer(response)
        extracted_answer = latex_postprocess(extracted_answer)  # <--- add this
        res[model] = extracted_answer
    return res

In [ ]:
sample_df['extracted_answer'] = sample_df.llm_output.apply(get_extracted_answer)

sample_df['accuracy_score'] = sample_df.apply(
    lambda r: get_accuracy_score(r.extracted_answer, r.answer),
    axis=1,
)

score_df = pd.DataFrame(sample_df.accuracy_score.tolist())

In [ ]:
score_df.notna().sum()

In [ ]:
score_df.fillna(False).mean().sort_values(ascending=False)